In [5]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.ollama import Ollama
from typing import Optional


class OllamaChat:
    def __init__(
        self,
        model: str = "qwen3:8b",
        system_prompt: Optional[str] = None,
        request_timeout: float = 120.0,
        context_window: int = 8000,
    ):
        self.llm = Ollama(
            model=model,
            request_timeout=request_timeout,
            context_window=context_window,
        )
        self.history: list[ChatMessage] = []

        if system_prompt:
            self.history.append(ChatMessage(role="system", content=system_prompt))

    def chat(self, user_message: str) -> str:
        """Send a message and get a response, maintaining history."""
        self.history.append(ChatMessage(role="user", content=user_message))

        response = self.llm.chat(self.history)
        assistant_message = response.message.content

        self.history.append(ChatMessage(role="assistant", content=assistant_message))

        return assistant_message

    def stream_chat(self, user_message: str):
        """Stream a response while maintaining history."""
        self.history.append(ChatMessage(role="user", content=user_message))

        full_response = ""
        for chunk in self.llm.stream_chat(self.history):
            delta = chunk.delta
            full_response += delta
            print(delta, end="", flush=True)

        print()  # newline after stream ends
        self.history.append(ChatMessage(role="assistant", content=full_response))

        return full_response

    def clear_history(self, keep_system_prompt: bool = True):
        """Clear conversation history, optionally keeping the system prompt."""
        if keep_system_prompt:
            self.history = [msg for msg in self.history if msg.role == "system"]
        else:
            self.history = []

    def print_history(self):
        """Print the full conversation history."""
        for msg in self.history:
            print(f"[{msg.role.upper()}]: {msg.content}\n")

In [6]:
# Basic usage
bot = OllamaChat(system_prompt="You are a helpful assistant.")

response = bot.chat("What is Python?")
print(response)

response = bot.chat("Can you give me a code example?")  # remembers previous context
print(response)

# Streaming
bot.stream_chat("Explain list comprehensions.")

# Inspect history
bot.print_history()

# Clear history but keep system prompt
bot.clear_history()

Python is a **high-level, interpreted programming language** known for its **readability**, **simplicity**, and **versatility**. It was created in the late 1980s by **Guido van Rossum** and officially released in 1991. Python emphasizes code readability with its use of **indentation** (white space) to define code blocks, making it beginner-friendly and easy to maintain.

### Key Features:
1. **Easy to Learn**: Simple syntax and English-like keywords reduce the learning curve.
2. **High-Level**: Abstracts low-level details, allowing developers to focus on solving problems.
3. **Dynamic Typing**: Variables are not explicitly declared, and types are determined at runtime.
4. **Cross-Platform**: Runs on Windows, macOS, Linux, and other operating systems.
5. **Open Source**: Free to use, modify, and distribute under the **Python Software Foundation License**.
6. **Extensive Libraries**: A vast standard library and third-party packages (e.g., NumPy, Pandas, TensorFlow) support diverse applic